# REM Sleep Behavior Disorder — Training Notebook

Dataset: 130 participants — PD (30), RB (50), HC (50)
Trains XGBoost and Random Forest sub-models, fuses them into a
REMEnsemble, and exports:
- `rem_ensemble.pkl`
- `modality_result.json`

## Step 1 — Imports

In [ ]:
import os
import json
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

import xgboost as xgb

warnings.filterwarnings('ignore')

OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Imports completed successfully')

## Step 2 — Load and Preprocess Data

In [ ]:
DATA_PATH = 'dataset.csv'

df = pd.read_csv(DATA_PATH)

# Strip all column name whitespace
df.columns = [c.strip() for c in df.columns]

print('Shape:', df.shape)
print(df.head())

In [ ]:
# ── Derive target from participant code prefix ─────────────────────────────
# PD = Parkinson's Disease, RB = REM Behaviour Disorder, HC = Healthy Control
df['label'] = df['Participant  code'].str[:2].map({'PD': 0, 'RB': 1, 'HC': 2})

print('Label distribution:')
print(df['label'].value_counts())

# ── Drop non-feature columns ───────────────────────────────────────────────
DROP_COLS = ['Participant  code', 'label']
X_raw = df.drop(columns=DROP_COLS)
y = df['label']

# ── Encode categoricals ────────────────────────────────────────────────────
CATEGORICAL_COLS = [
    'Gender',
    'Positive  history  of  Parkinson  disease  in  family',
    'Antidepressant  therapy',
    'Antiparkinsonian  medication',
    'Antipsychotic  medication',
    'Benzodiazepine  medication',
]

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X_raw[col] = le.fit_transform(X_raw[col].astype(str).str.strip())

# ── Remaining string columns to numeric ───────────────────────────────────
for col in X_raw.select_dtypes(include='object').columns:
    X_raw[col] = pd.to_numeric(X_raw[col], errors='coerce')

X_raw = X_raw.fillna(X_raw.median())

print('Features shape:', X_raw.shape)
print('Dtypes after encoding:')
print(X_raw.dtypes.value_counts())

## Step 3 — Train / Val / Test Split

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_raw, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print('Train:', X_train.shape)
print('Val:  ', X_val.shape)
print('Test: ', X_test.shape)

## Step 4 — Train XGBoost Sub-model

In [ ]:
scaler_xgb = StandardScaler()
X_train_xgb = scaler_xgb.fit_transform(X_train)
X_val_xgb   = scaler_xgb.transform(X_val)
X_test_xgb  = scaler_xgb.transform(X_test)

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.5,
    random_state=42,
    tree_method='hist',
    eval_metric='mlogloss',
)

xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_val_xgb, y_val)],
    verbose=False
)

xgb_preds = xgb_model.predict(X_test_xgb)
xgb_acc   = accuracy_score(y_test, xgb_preds)
xgb_f1    = f1_score(y_test, xgb_preds, average='weighted', zero_division=0)

print('XGBoost Test Accuracy:', round(xgb_acc, 4))
print('XGBoost Test F1:      ', round(xgb_f1, 4))
print(classification_report(y_test, xgb_preds, zero_division=0))

## Step 5 — Train Random Forest Sub-model

In [ ]:
scaler_rf = StandardScaler()
X_train_rf = scaler_rf.fit_transform(X_train)
X_val_rf   = scaler_rf.transform(X_val)
X_test_rf  = scaler_rf.transform(X_test)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train)

rf_preds = rf_model.predict(X_test_rf)
rf_acc   = accuracy_score(y_test, rf_preds)
rf_f1    = f1_score(y_test, rf_preds, average='weighted', zero_division=0)

print('OOB Score:                  ', round(rf_model.oob_score_, 4))
print('Random Forest Test Accuracy:', round(rf_acc, 4))
print('Random Forest Test F1:      ', round(rf_f1, 4))
print(classification_report(y_test, rf_preds, zero_division=0))

## Step 6 — Build REMEnsemble

In [ ]:
class _SubModel:
    """Thin wrapper — inference_rem.py accesses sub_model.scaler and sub_model.model."""
    def __init__(self, model, scaler):
        self.model  = model
        self.scaler = scaler

    def predict(self, X):
        return self.model.predict(self.scaler.transform(X))

    def predict_proba(self, X):
        return self.model.predict_proba(self.scaler.transform(X))


class REMEnsemble:
    """
    Fused ensemble consumed by inference_rem.py.
    Required attributes:
        .feature_names  – list[str]
        .models         – dict[str, _SubModel]
        .fusion_method  – str
    """
    def __init__(self, models, feature_names, fusion_method='voting'):
        self.models        = models
        self.feature_names = feature_names
        self.fusion_method = fusion_method

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        proba_list = [sub.predict_proba(X) for sub in self.models.values()]
        return np.mean(proba_list, axis=0)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        preds = np.column_stack([sub.predict(X) for sub in self.models.values()])
        return np.apply_along_axis(
            lambda row: np.bincount(row.astype(int)).argmax(),
            axis=1,
            arr=preds
        )


FUSION_METHOD = 'voting'

ensemble = REMEnsemble(
    models={
        'XGBoost':      _SubModel(xgb_model, scaler_xgb),
        'RandomForest': _SubModel(rf_model,  scaler_rf),
    },
    feature_names=list(X_raw.columns),
    fusion_method=FUSION_METHOD,
)

ensemble_preds = ensemble.predict(
    pd.DataFrame(X_test.values, columns=X_raw.columns)
)

ensemble_acc = accuracy_score(y_test, ensemble_preds)
ensemble_f1  = f1_score(
    y_test, ensemble_preds,
    average='weighted',
    zero_division=0
)

print('Ensemble Test Accuracy:', round(ensemble_acc, 4))
print('Ensemble Test F1:      ', round(ensemble_f1, 4))
print(classification_report(y_test, ensemble_preds, zero_division=0))

## Step 7 — Export Artifacts